# Music Annotation Project Notebook 70
## RAW vs 59R5_MERGED × Pilot 09 vs V8.8 Registered-Replay Correction-Engine Crossover

**Runtime:** CPU. Select **Runtime → Run all**.

### Important correction to the experimental label
The later operational comparator available on the **same Pilot 09 page population** is the frozen **V8.8 registered-replay correction engine** from Notebooks 53–56. It is not merely a different inpainting/restoration algorithm: compared with Pilot 09 it also embodies the later M5 residual-mask/correction-engine policy. Notebook 70 therefore estimates a **correction-engine effect**, not a pure restorer-only effect.

The older `R3_learned_unet` is a true learned restorer, but its frozen held-out models belong to the V8.6/V8.7 development source-group experiment and are not automatically a valid frozen engine for the exact Pilot 09 population. Notebook 70 does not silently repurpose them.

### Primary 2×2 question
On the exact Notebook 69 / Notebook 67 Pilot population, and using clean pages **only after candidate lock**, compare:

| Semantic acceptance unit | Pilot 09 correction engine | Later frozen engine |
|---|---|---|
| RAW focal component | `RAW × PILOT09` | `RAW × V8.8` |
| Frozen 59R5 merged members | `MERGED × PILOT09` | `MERGED × V8.8` |

### What changes and what does not
- **Acceptance-unit variable:** exact focal Pilot component versus union of its frozen 59R5 merged member components.
- **Engine variable:** saved Pilot 09 proposal versus saved V8.8 registered-replay corrected page.
- **Authoritative mask:** unchanged Pilot 09 intervention mask.
- **Outside authority:** byte-for-byte observed page in every arm.
- **Semantic model:** not rerun.
- **59R5 merge:** not recomputed; Notebook 69's persisted group membership is reused.
- **Clean truth:** prohibited until all four candidate-value hashes are locked.
- **No source outputs are modified.**

### Primary outcome interpretation
Within the SAME merged evaluation region for each focal candidate:
- lower `false_inserted_ink` = less residual annotation / spurious dark material relative to clean truth;
- lower `missing_ink` = less genuine clean printed ink lost;
- higher `ink_f1` and `edge_f1` = better print/edge fidelity;
- lower MAE = closer tonal agreement to clean truth.

This design makes the two main effects and their interaction separately inspectable rather than changing grouping and correction engine at the same time and calling the combined result an improvement.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy scipy scikit-image matplotlib

## 1. Configuration, bounded source resolution, and output tree

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, gc, sys, platform
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage.metrics import structural_similarity as ssim

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
N69_ROOT=ROOT/'notebook69_merged_semantic_shadow'
N53_ROOT=ROOT/'observed_print_classifier_v8_8_notebook53R3'
PILOT_ROOT=ROOT/'production_pilot'/'runs'
OUT_ROOT=ROOT/'notebook70_context_engine_crossover'

# Exact overrides are preferred for the definitive run.
NOTEBOOK69_RUN_OVERRIDE=None
NOTEBOOK53_RUN_OVERRIDE=None
PILOT09_RUN_OVERRIDE=None

SEED=70070
BOOTSTRAP_RESAMPLES=10000
MIN_COMPONENT_AREA=20
VISUAL_CASES_PER_STRATUM=12
CROP_MARGIN=56

SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}

def require(condition,message):
    if not bool(condition): raise RuntimeError(message)

def sha256_file(path):
    path=Path(path); require(path.is_file(),f'Missing file: {path}')
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()

def atomic_json(path,value):
    path=Path(path); tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str)); tmp.replace(path)

def atomic_csv(frame,path):
    path=Path(path); tmp=Path(str(path)+'.tmp'); frame.to_csv(tmp,index=False); tmp.replace(path)

def direct_complete(root,required):
    root=Path(root)
    if not root.exists(): return []
    return sorted([p for p in root.iterdir() if p.is_dir() and all((p/r).exists() for r in required)],key=lambda p:p.stat().st_mtime)

def image_map(folder):
    folder=Path(folder)
    return {p.stem:p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if folder.exists() else {}

req69=['tables/merged_representation_manifest.csv','tables/raw_vs_merged_paired_components.csv','source_identity.json','notebook69_verdict.json']
if NOTEBOOK69_RUN_OVERRIDE: N69=Path(NOTEBOOK69_RUN_OVERRIDE)
else:
    hits=direct_complete(N69_ROOT,req69); require(hits,f'No complete Notebook 69 run directly below {N69_ROOT}'); N69=hits[-1]

sid69=json.loads((N69/'source_identity.json').read_text())
pilot_record=sid69.get('pilot09_run') or sid69.get('pilot_09_run') or sid69.get('pilot_run')
if PILOT09_RUN_OVERRIDE: PILOT=Path(PILOT09_RUN_OVERRIDE)
elif pilot_record and Path(pilot_record).exists(): PILOT=Path(pilot_record)
else:
    hits=direct_complete(PILOT_ROOT,['pilot_manifest.json','tables/component_evidence.csv','routing','proposals','audit']); require(hits,'No complete Pilot 09 run found'); PILOT=hits[-1]

req53=['stage_f_verdict.json','experiment_identity.json','pages','tables/page_equivalence.csv']
if NOTEBOOK53_RUN_OVERRIDE: N53=Path(NOTEBOOK53_RUN_OVERRIDE)
else:
    hits=direct_complete(N53_ROOT,req53); require(hits,f'No complete Notebook 53R3 run directly below {N53_ROOT}'); N53=hits[-1]

v53=json.loads((N53/'stage_f_verdict.json').read_text()); id53=json.loads((N53/'experiment_identity.json').read_text())
require(v53.get('verdict')=='PASS_STAGE_F_REGRESSION_COMPATIBILITY','Notebook 53R3 did not pass regression compatibility.')
require(id53.get('mode')=='REGISTERED_REPLAY','Notebook 53 is not registered replay mode.')

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook70_2x2_crossover'
OUT=OUT_ROOT/RUN; TABLES=OUT/'tables'; AUDIT=OUT/'audit'; VIS=OUT/'visual_audit'; LOCK=OUT/'candidate_lock'; UPLOAD=OUT/'upload'
for p in [OUT,TABLES,AUDIT,VIS,LOCK,UPLOAD]: p.mkdir(parents=True,exist_ok=False)

print('Notebook 69:',N69)
print('Pilot 09:',PILOT)
print('V8.8 / Notebook 53R3:',N53)
print('Output:',OUT)

## 2. Exact candidate roster and frozen 59R5 merged membership

In [ ]:
manifest=pd.read_csv(N69/'tables'/'merged_representation_manifest.csv',low_memory=False)
paired=pd.read_csv(N69/'tables'/'raw_vs_merged_paired_components.csv',low_memory=False)
components=pd.read_csv(PILOT/'tables'/'component_evidence.csv',low_memory=False)
for d,name in [(manifest,'manifest'),(paired,'paired'),(components,'components')]:
    require({'page','component'}.issubset(d.columns),f'{name} lacks page/component.')
    d['page']=d.page.astype(str); d['component']=pd.to_numeric(d.component,errors='raise').astype(int)

member_col=next((c for c in ['member_components','merged_member_components','members','member_ids'] if c in manifest.columns),None)
require(member_col is not None,f'Merged manifest lacks member IDs. Columns={manifest.columns.tolist()}')
def parse_members(v):
    if pd.isna(v): return []
    text=str(v).replace(',','|').replace(';','|')
    return sorted({int(float(x.strip())) for x in text.split('|') if x.strip()})
manifest['member_ids']=manifest[member_col].map(parse_members)
manifest=manifest[['page','component','member_ids']].drop_duplicates(['page','component'])
require(len(manifest)==len(components),f'Notebook69 manifest/Pilot component count mismatch: {len(manifest)} vs {len(components)}')

roster=components[['page','component','x','y','bbox_w','bbox_h']].merge(manifest,on=['page','component'],how='left',validate='one_to_one')
require(roster.member_ids.notna().all(),'Missing 59R5 member group for one or more Pilot components.')
roster['member_ids']=[sorted(set((v or [])+[int(c)])) for v,c in zip(roster.member_ids,roster.component)]
roster['member_count']=roster.member_ids.map(len)
roster['candidate_id']=[f'{p}:C{c}' for p,c in zip(roster.page,roster.component)]
roster['group_id']=[hashlib.sha256((p+'|'+'|'.join(map(str,m))).encode()).hexdigest()[:20] for p,m in zip(roster.page,roster.member_ids)]
atomic_csv(roster.assign(member_components=roster.member_ids.map(lambda x:'|'.join(map(str,x)))).drop(columns='member_ids'),TABLES/'candidate_roster.csv')
print('Candidates:',len(roster),'Pages:',roster.page.nunique(),'Expanded merged groups:',int((roster.member_count>1).sum()))
print(roster.member_count.describe())

## 3. Load observed pages, authoritative Pilot masks, exact component labels, and both saved correction engines

The component mask is reconstructed from the exact connected-component labels of the Pilot 09 authoritative mask. Bounding boxes are used only as an identity cross-check, not as correction authority.

In [ ]:
proposal09=image_map(PILOT/'proposals')
clean_dir=ROOT/'source'/'clean'
clean_paths=image_map(clean_dir)  # Paths only. Pixels are NOT read before candidate lock.

page_data={}; identity_rows=[]
for page in sorted(roster.page.unique()):
    rp=PILOT/'audit'/f'{page}_input_resolution.json'; require(rp.exists(),f'{page}: missing Pilot input-resolution audit')
    res=json.loads(rp.read_text()); op=Path(str(res.get('observed_path',''))); require(op.exists(),f'{page}: observed path unavailable: {op}')
    obs=cv2.imread(str(op),0); require(obs is not None,f'{page}: observed unreadable')
    p09=proposal09.get(page); require(p09 is not None,f'{page}: Pilot09 proposal missing')
    im09=cv2.imread(str(p09),0); require(im09 is not None and im09.shape==obs.shape,f'{page}: invalid Pilot09 proposal')
    p88=N53/'pages'/page/'corrected.png'; require(p88.exists(),f'{page}: V8.8 registered-replay corrected page missing: {p88}')
    im88=cv2.imread(str(p88),0); require(im88 is not None and im88.shape==obs.shape,f'{page}: invalid V8.8 corrected page')
    obs88=cv2.imread(str(N53/'pages'/page/'observed.png'),0); require(obs88 is not None and np.array_equal(obs88,obs),f'{page}: Pilot09 and V8.8 observed pages are not byte-identical')
    route=PILOT/'routing'/f'{page}.npz'; require(route.exists(),f'{page}: Pilot routing missing')
    with np.load(route) as z:
        require('mask' in z.files,f'{page}: routing lacks mask'); auth=(z['mask']>0)
    require(auth.shape==obs.shape,f'{page}: authority shape mismatch')
    # Pilot 09 connected-component semantics use 8-connectivity on the authoritative mask.
    n,labels,stats,_=cv2.connectedComponentsWithStats(auth.astype(np.uint8),8)
    f=components[components.page==page]
    for _,r in f.iterrows():
        cid=int(r.component); require(0<cid<n,f'{page}: component {cid} outside connected-component label range 1..{n-1}')
        cm=(labels==cid); require(int(cm.sum())>=MIN_COMPONENT_AREA,f'{page}:C{cid} exact component smaller than retained minimum')
        x=int(stats[cid,cv2.CC_STAT_LEFT]); y=int(stats[cid,cv2.CC_STAT_TOP]); w=int(stats[cid,cv2.CC_STAT_WIDTH]); h=int(stats[cid,cv2.CC_STAT_HEIGHT])
        require((x,y,w,h)==(int(r.x),int(r.y),int(r.bbox_w),int(r.bbox_h)),f'{page}:C{cid} component bbox identity mismatch')
    require(np.array_equal(im09[~auth],obs[~auth]),f'{page}: Pilot09 proposal changes pixels outside authority')
    require(np.array_equal(im88[~auth],obs[~auth]),f'{page}: V8.8 output changes pixels outside Pilot authority')
    page_data[page]={'obs':obs,'pilot09':im09,'v88':im88,'auth':auth,'labels':labels,'stats':stats,'observed_path':op,'pilot09_path':p09,'v88_path':p88}
    identity_rows.append({'page':page,'observed_path':str(op),'observed_sha256':sha256_file(op),'pilot09_path':str(p09),'pilot09_sha256':sha256_file(p09),'v88_path':str(p88),'v88_sha256':sha256_file(p88),'clean_path_declared':str(clean_paths.get(page,'')),'clean_exists':page in clean_paths})
identity=pd.DataFrame(identity_rows); atomic_csv(identity,AUDIT/'page_source_identity.csv')
require(identity.clean_exists.all(),f'Clean counterparts missing by stem: {identity.loc[~identity.clean_exists,"page"].tolist()}')
print('Loaded and identity-verified pages before truth unlock:',len(page_data))

## 4. Construct and LOCK the four candidate-value vectors before clean truth

For every focal component, all four arms are scored on the exact same **merged member union**. RAW arms modify only the focal component and intentionally leave the other group members in their observed state. MERGED arms apply the selected engine output to all member-component pixels.

Only the pixels inside the merged member union are hashed into the candidate lock because that is the fixed region on which the four arms can differ. Clean pixels have not yet been read.

In [ ]:
def component_union(page,ids):
    z=page_data[page]; m=np.zeros(z['auth'].shape,bool)
    for cid in ids:
        cid=int(cid); require(0<cid<z['stats'].shape[0],f'{page}: invalid member component {cid}'); m|=(z['labels']==cid)
    return m & z['auth']

def arm_values(page,focal,members,engine_key,merged):
    z=page_data[page]; eval_mask=component_union(page,members); accept_mask=eval_mask if merged else component_union(page,[focal])
    out=z['obs'].copy(); source=z[engine_key]; out[accept_mask]=source[accept_mask]
    require(np.array_equal(out[~z['auth']],z['obs'][~z['auth']]),f'{page}: outside-authority drift')
    return out[eval_mask].copy(),eval_mask,accept_mask

lock_rows=[]
for i,r in roster.iterrows():
    p=r.page; c=int(r.component); members=r.member_ids
    eval_mask=component_union(p,members); require(eval_mask.any(),f'{r.candidate_id}: empty merged evaluation mask')
    for unit,merged in [('RAW',False),('MERGED',True)]:
        for engine,key in [('PILOT09','pilot09'),('V8_8','v88')]:
            values,_,accept=arm_values(p,c,members,key,merged)
            lock_rows.append({'candidate_id':r.candidate_id,'page':p,'component':c,'group_id':r.group_id,'member_count':len(members),'acceptance_unit':unit,'engine':engine,'arm':f'{unit}__{engine}','evaluation_pixels':int(eval_mask.sum()),'accepted_pixels':int(accept.sum()),'candidate_values_sha256':array_hash(values),'observed_sha256':array_hash(page_data[p]['obs']),'authority_sha256':array_hash(page_data[p]['auth'].astype(np.uint8))})
lock_df=pd.DataFrame(lock_rows).sort_values(['candidate_id','acceptance_unit','engine']).reset_index(drop=True)
require(len(lock_df)==4*len(roster),'Candidate lock does not contain four arms per focal candidate.')
atomic_csv(lock_df,LOCK/'candidate_value_inventory.csv')
lock_id=hashlib.sha256(lock_df.to_csv(index=False).encode()).hexdigest()
atomic_json(LOCK/'candidate_lock.json',{'version':'notebook70-candidate-lock-v1','lock_id':lock_id,'truth_loaded':False,'rows':len(lock_df),'candidates':len(roster),'pages':int(roster.page.nunique()),'engines':{'PILOT09':'saved Pilot09 proposal','V8_8':'Notebook53R3 registered-replay corrected page'},'acceptance_units':['RAW','MERGED'],'created_utc':datetime.now(timezone.utc).isoformat()})
print('CANDIDATE LOCK:',lock_id,'rows:',len(lock_df))

## 5. Truth unlock: load exact clean counterparts only after candidate lock

Notebook 70 requires exact same-shape clean pages by page stem under `source/clean`. It does not resize, align, warp, substitute, or search for a visually similar clean page after seeing candidate outcomes.

In [ ]:
clean={}; truth_rows=[]
for page in sorted(page_data):
    cp=clean_paths[page]; im=cv2.imread(str(cp),0); require(im is not None,f'{page}: clean unreadable')
    require(im.shape==page_data[page]['obs'].shape,f'{page}: clean shape {im.shape} != observed {page_data[page]["obs"].shape}; fail closed rather than register post hoc')
    clean[page]=im; truth_rows.append({'page':page,'clean_path':str(cp),'clean_sha256':sha256_file(cp),'shape':str(im.shape)})
atomic_csv(pd.DataFrame(truth_rows),AUDIT/'clean_truth_identity_after_lock.csv')
atomic_json(AUDIT/'truth_firewall.json',{'candidate_lock_id':lock_id,'candidate_lock_created_before_clean_pixels_loaded':True,'clean_used_for_candidate_construction':False,'clean_used_for_grouping':False,'clean_used_for_engine_output':False,'clean_role':'post-lock evaluation only'})
print('Truth unlocked for pages:',len(clean))

## 6. Verify candidate replay against lock and score clean truth

In [ ]:
def ink(im): return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edges(im): return cv2.Canny(im,70,170)>0

def tolerant_prf(ref,pred,region,tol=1):
    ref=ref&region; pred=pred&region
    if not ref.any(): return np.nan,np.nan,np.nan
    k=np.ones((2*tol+1,2*tol+1),np.uint8)
    prec=float(np.sum(pred&(cv2.dilate(ref.astype(np.uint8),k)>0))/max(int(pred.sum()),1))
    rec=float(np.sum(ref&(cv2.dilate(pred.astype(np.uint8),k)>0))/int(ref.sum()))
    f=float(2*prec*rec/max(prec+rec,1e-12)); return prec,rec,f

def metric_row(clean_im,pred,obs,region):
    require(region.any(),'Empty scoring region')
    d=np.abs(clean_im.astype(np.float32)-pred.astype(np.float32)); mse=float(np.mean((clean_im[region].astype(float)-pred[region].astype(float))**2))
    _,sm=ssim(clean_im,pred,data_range=255,full=True)
    ci=ink(clean_im); pi=ink(pred); ce=edges(clean_im); pe=edges(pred)
    ip,ir,if1=tolerant_prf(ci,pi,region,1); ep,er,ef1=tolerant_prf(ce,pe,region,1)
    return {'evaluation_pixels':int(region.sum()),'mae':float(d[region].mean()),'rmse':float(np.sqrt(mse)),'ssim':float(sm[region].mean()),'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_precision':ep,'edge_recall':er,'edge_f1':ef1,'false_inserted_ink':int((pi&~ci&region).sum()),'missing_ink':int((ci&~pi&region).sum()),'changed_from_observed':int(((pred!=obs)&region).sum())}

locked={(r.candidate_id,r.acceptance_unit,r.engine):r.candidate_values_sha256 for _,r in lock_df.iterrows()}
score_rows=[]
for ix,r in roster.iterrows():
    p=r.page; c=int(r.component); members=r.member_ids; eval_mask=component_union(p,members); z=page_data[p]
    for unit,merged in [('RAW',False),('MERGED',True)]:
        for engine,key in [('PILOT09','pilot09'),('V8_8','v88')]:
            values,_,accept=arm_values(p,c,members,key,merged)
            require(array_hash(values)==locked[(r.candidate_id,unit,engine)],f'{r.candidate_id}/{unit}/{engine}: candidate replay differs from pre-truth lock')
            pred=z['obs'].copy(); pred[eval_mask]=values
            m=metric_row(clean[p],pred,z['obs'],eval_mask)
            score_rows.append({'candidate_id':r.candidate_id,'page':p,'component':c,'group_id':r.group_id,'member_count':len(members),'acceptance_unit':unit,'engine':engine,'arm':f'{unit}__{engine}','accepted_pixels':int(accept.sum()),**m})
    if (ix+1)%250==0 or ix+1==len(roster): print(f'Scored {ix+1}/{len(roster)} candidates')
metrics=pd.DataFrame(score_rows); atomic_csv(metrics,TABLES/'candidate_2x2_clean_truth_metrics.csv')
require(len(metrics)==4*len(roster),'Incomplete score table')
print('Scored rows:',len(metrics))

## 7. Primary 2×2 estimands

Positive **benefit** always means improvement:
- MAE / false inserted ink / missing ink: comparator minus treatment;
- ink F1 / edge F1 / SSIM: treatment minus comparator.

The three paired effects are:
1. **Grouping effect with engine held fixed:** MERGED − RAW.
2. **Engine effect with grouping held fixed:** V8.8 − Pilot09.
3. **Interaction:** whether the grouping benefit is larger under V8.8 than under Pilot09.

Inference is clustered by page through a page bootstrap. Candidate-level means remain descriptive because merged groups overlap.

In [ ]:
METRICS=['mae','ssim','ink_f1','edge_f1','false_inserted_ink','missing_ink']
LOWER={'mae','false_inserted_ink','missing_ink'}
def orient(metric,treatment,control): return (control-treatment) if metric in LOWER else (treatment-control)

def candidate_wide(metric): return metrics.pivot(index=['candidate_id','page','component','group_id','member_count'],columns='arm',values=metric).reset_index()

def page_bootstrap(frame,value_col,n=BOOTSTRAP_RESAMPLES,seed=SEED):
    p=frame.groupby('page')[value_col].mean().dropna()
    if p.empty:return np.nan,np.nan,np.nan,0
    vals=p.to_numpy(float); rng=np.random.default_rng(seed); boot=np.array([rng.choice(vals,len(vals),replace=True).mean() for _ in range(n)])
    return float(vals.mean()),float(np.quantile(boot,.025)),float(np.quantile(boot,.975)),int(len(vals))

rows=[]; candidate_effects=[]
for metric in METRICS:
    w=candidate_wide(metric)
    w['grouping_pilot09']=orient(metric,w['MERGED__PILOT09'],w['RAW__PILOT09'])
    w['grouping_v88']=orient(metric,w['MERGED__V8_8'],w['RAW__V8_8'])
    w['engine_raw']=orient(metric,w['RAW__V8_8'],w['RAW__PILOT09'])
    w['engine_merged']=orient(metric,w['MERGED__V8_8'],w['MERGED__PILOT09'])
    w['interaction']=w['grouping_v88']-w['grouping_pilot09']
    w['metric']=metric; candidate_effects.append(w)
    for effect in ['grouping_pilot09','grouping_v88','engine_raw','engine_merged','interaction']:
        mean,lo,hi,npages=page_bootstrap(w[['page',effect]].rename(columns={effect:'value'}),'value',seed=SEED+len(rows))
        v=w[effect].dropna()
        rows.append({'metric':metric,'effect':effect,'benefit_mean_page_clustered':mean,'ci95_low':lo,'ci95_high':hi,'pages':npages,'candidate_rows':int(len(v)),'candidate_wins':int((v>0).sum()),'candidate_losses':int((v<0).sum()),'candidate_ties':int((v==0).sum())})

effects=pd.DataFrame(rows); atomic_csv(effects,TABLES/'paired_2x2_effects_page_bootstrap.csv')
all_effects=pd.concat(candidate_effects,ignore_index=True); atomic_csv(all_effects,TABLES/'candidate_effects_long_metric.csv')
summary=metrics.groupby(['acceptance_unit','engine'],as_index=False).agg(candidates=('candidate_id','nunique'),pages=('page','nunique'),mae=('mae','mean'),ssim=('ssim','mean'),ink_f1=('ink_f1','mean'),edge_f1=('edge_f1','mean'),false_inserted_ink=('false_inserted_ink','sum'),missing_ink=('missing_ink','sum'),changed_from_observed=('changed_from_observed','sum'))
atomic_csv(summary,TABLES/'arm_summary.csv')
display(summary); display(effects)

## 8. Expanded-group-only sensitivity

Rows with `member_count == 1` cannot distinguish RAW from MERGED. The primary table retains the complete population for lineage, while this table asks the grouping question only where 59R5 actually expands the focal component.

In [ ]:
expanded_ids=set(roster.loc[roster.member_count>1,'candidate_id'])
expanded=metrics[metrics.candidate_id.isin(expanded_ids)].copy()
require(len(expanded)>0,'No expanded 59R5 groups available.')
rows=[]
for metric in METRICS:
    w=expanded.pivot(index=['candidate_id','page','component','group_id','member_count'],columns='arm',values=metric).reset_index()
    for name,treat,control in [('grouping_pilot09','MERGED__PILOT09','RAW__PILOT09'),('grouping_v88','MERGED__V8_8','RAW__V8_8'),('engine_raw','RAW__V8_8','RAW__PILOT09'),('engine_merged','MERGED__V8_8','MERGED__PILOT09')]: w[name]=orient(metric,w[treat],w[control])
    w['interaction']=w['grouping_v88']-w['grouping_pilot09']
    for effect in ['grouping_pilot09','grouping_v88','engine_raw','engine_merged','interaction']:
        mean,lo,hi,npages=page_bootstrap(w[['page',effect]].rename(columns={effect:'value'}),'value',seed=SEED+100+len(rows)); v=w[effect].dropna()
        rows.append({'metric':metric,'effect':effect,'expanded_candidates':len(w),'benefit_mean_page_clustered':mean,'ci95_low':lo,'ci95_high':hi,'pages':npages,'wins':int((v>0).sum()),'losses':int((v<0).sum()),'ties':int((v==0).sum())})
expanded_effects=pd.DataFrame(rows); atomic_csv(expanded_effects,TABLES/'expanded_groups_2x2_effects.csv'); display(expanded_effects)

## 9. Page-level stability and clean-truth harm flags

A grouping/engine improvement is not treated as benign if it systematically lowers false inserted ink while increasing missing clean ink. Both axes are reported separately.

In [ ]:
page_summary=metrics.groupby(['page','arm'],as_index=False).agg(candidates=('candidate_id','nunique'),mae=('mae','mean'),ink_f1=('ink_f1','mean'),edge_f1=('edge_f1','mean'),false_inserted_ink=('false_inserted_ink','sum'),missing_ink=('missing_ink','sum'))
atomic_csv(page_summary,TABLES/'metrics_by_page_arm.csv')
p=page_summary.pivot(index='page',columns='arm',values=['false_inserted_ink','missing_ink','ink_f1','edge_f1','mae'])
flags=[]
for page in p.index:
    def V(metric,arm): return float(p.loc[page,(metric,arm)])
    flags.append({'page':page,
        'merged_v88_vs_raw_pilot09_false_inserted_benefit':V('false_inserted_ink','RAW__PILOT09')-V('false_inserted_ink','MERGED__V8_8'),
        'merged_v88_vs_raw_pilot09_missing_ink_benefit':V('missing_ink','RAW__PILOT09')-V('missing_ink','MERGED__V8_8'),
        'merged_v88_improves_residue_but_harms_print':(V('false_inserted_ink','MERGED__V8_8')<V('false_inserted_ink','RAW__PILOT09')) and (V('missing_ink','MERGED__V8_8')>V('missing_ink','RAW__PILOT09'))})
flags=pd.DataFrame(flags); atomic_csv(flags,TABLES/'page_harm_tradeoff_flags.csv'); print(flags.merged_v88_improves_residue_but_harms_print.value_counts())

## 10. Visual audit: strongest gains, strongest losses, and interactions

Each crop shows **Observed, Clean, RAW×Pilot09, MERGED×Pilot09, RAW×V8.8, MERGED×V8.8**. Clean truth is displayed only in this post-lock evaluation stage.

In [ ]:
def crop_bounds(mask,margin=CROP_MARGIN):
    ys,xs=np.where(mask); require(len(xs),'Empty crop mask'); h,w=mask.shape
    return max(0,int(xs.min())-margin),max(0,int(ys.min())-margin),min(w,int(xs.max())+1+margin),min(h,int(ys.max())+1+margin)
def full_arm(page,focal,members,key,merged):
    z=page_data[page]; em=component_union(page,members); am=em if merged else component_union(page,[focal]); out=z['obs'].copy(); out[am]=z[key][am]; return out,em

def render_case(candidate_id,label,metric,effect_value):
    r=roster[roster.candidate_id==candidate_id].iloc[0]; page=r.page; cid=int(r.component); members=r.member_ids
    a,em=full_arm(page,cid,members,'pilot09',False); b,_=full_arm(page,cid,members,'pilot09',True); c,_=full_arm(page,cid,members,'v88',False); d,_=full_arm(page,cid,members,'v88',True)
    x0,y0,x1,y1=crop_bounds(em); ims=[page_data[page]['obs'],clean[page],a,b,c,d]; titles=['Observed','Clean truth','RAW × Pilot09','MERGED × Pilot09','RAW × V8.8','MERGED × V8.8']
    fig,axes=plt.subplots(2,3,figsize=(15,10));
    for ax,im,title in zip(axes.ravel(),ims,titles): ax.imshow(im[y0:y1,x0:x1],cmap='gray',vmin=0,vmax=255); ax.set_title(title); ax.axis('off')
    fig.suptitle(f'{candidate_id} | {label} | {metric} benefit={effect_value:.5f} | members={members}',fontsize=11); plt.tight_layout(rect=[0,0,1,.94])
    folder=VIS/label; folder.mkdir(parents=True,exist_ok=True); path=folder/f'{candidate_id.replace(":","_")}_{metric}.png'; fig.savefig(path,dpi=150,bbox_inches='tight'); plt.close(fig); return path

# Build candidate effects for two interpretable endpoints.
vis_rows=[]
for metric in ['false_inserted_ink','missing_ink','ink_f1','edge_f1']:
    w=metrics.pivot(index=['candidate_id','page'],columns='arm',values=metric).reset_index()
    w['combined']=orient(metric,w['MERGED__V8_8'],w['RAW__PILOT09'])
    w['grouping_under_v88']=orient(metric,w['MERGED__V8_8'],w['RAW__V8_8'])
    for label,col,ascending in [('COMBINED_BEST','combined',False),('COMBINED_WORST','combined',True),('GROUPING_V88_BEST','grouping_under_v88',False),('GROUPING_V88_WORST','grouping_under_v88',True)]:
        q=w.sort_values(col,ascending=ascending).head(VISUAL_CASES_PER_STRATUM)
        for _,x in q.iterrows():
            path=render_case(x.candidate_id,label,metric,float(x[col])); vis_rows.append({'candidate_id':x.candidate_id,'page':x.page,'label':label,'metric':metric,'benefit':float(x[col]),'render_path':str(path),'sha256':sha256_file(path)})
visual=pd.DataFrame(vis_rows).drop_duplicates(['candidate_id','label','metric']); atomic_csv(visual,TABLES/'visual_audit_manifest.csv')
print('Visual audit images:',len(visual),'in',VIS)

## 11. Predeclared interpretation, not a promotion rule

Notebook 70 is a controlled development/diagnostic crossover. It does not promote 59R5 grouping, V8.8, or any automatic correction tier.

The outcome should be read causally:
- **Grouping positive, engine neutral:** coordinated component acceptance matters more than the correction engine.
- **Engine positive, grouping neutral:** later correction engineering matters; semantic grouping adds little reconstruction value.
- **Both positive, interaction near zero:** benefits are approximately additive.
- **Positive interaction:** the V8.8 engine benefits disproportionately when coherent merged members are accepted together.
- **False-inserted improvement with missing-ink harm:** cosmetic annotation removal is being bought by genuine-print loss and is not a safe win.
- **Neither main effect positive:** the remaining bottleneck is neither simple component grouping nor the V8.8 correction-engine substitution.

In [ ]:
# Compact machine-readable conclusion from signs/CIs only. No threshold tuning.
key=effects.set_index(['metric','effect'])
def state(metric,effect):
    r=key.loc[(metric,effect)]
    if float(r.ci95_low)>0:return 'POSITIVE_CI'
    if float(r.ci95_high)<0:return 'NEGATIVE_CI'
    return 'INCONCLUSIVE_CI'
verdict={
 'version':'notebook70-2x2-v1',
 'candidate_lock_id':lock_id,
 'valid_run':True,
 'development_only':True,
 'automatic_authority':False,
 'grouping_effect_pilot09':{m:state(m,'grouping_pilot09') for m in METRICS},
 'grouping_effect_v88':{m:state(m,'grouping_v88') for m in METRICS},
 'engine_effect_raw':{m:state(m,'engine_raw') for m in METRICS},
 'engine_effect_merged':{m:state(m,'engine_merged') for m in METRICS},
 'interaction':{m:state(m,'interaction') for m in METRICS},
 'safety_priority':'false_inserted_ink must not improve by materially worsening missing clean ink / structural fidelity',
 'truth_role':'post-lock scoring only',
 'later_engine_interpretation':'V8.8 registered-replay correction engine; not a pure restorer-only comparison',
}
atomic_json(OUT/'notebook70_verdict.json',verdict)
print(json.dumps(verdict,indent=2))

## 12. Identity, read-only audit, executive summary, and upload package

In [ ]:
read_only={'source_runs_modified':False,'semantic_inference_rerun':False,'semantic_model_retrained':False,'59R5_merging_recomputed':False,'pilot09_engine_recomputed':False,'v88_engine_recomputed':False,'clean_truth_used_before_candidate_lock':False,'outside_authority_changes_allowed':False}
atomic_json(AUDIT/'read_only_and_truth_firewall_audit.json',read_only)
identity={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'notebook70-2x2-v1','notebook69_run':str(N69),'pilot09_run':str(PILOT),'notebook53R3_run':str(N53),'notebook53_version':id53.get('version'),'notebook53_mode':id53.get('mode'),'candidate_lock_id':lock_id,'candidate_count':len(roster),'page_count':int(roster.page.nunique()),'expanded_group_candidates':int((roster.member_count>1).sum()),'clean_directory':str(clean_dir),'software':{'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}}
atomic_json(OUT/'experiment_identity.json',identity)

lines=['NOTEBOOK 70 — RAW vs 59R5_MERGED × PILOT09 vs V8.8 CLEAN-TRUTH CROSSOVER','',f'Run: {OUT}',f'Candidates: {len(roster)}',f'Pages: {roster.page.nunique()}',f'Expanded merged candidates: {(roster.member_count>1).sum()}',f'Candidate lock: {lock_id}','','Important: V8.8 is the later registered-replay correction engine, not a pure restorer-only substitution.','Clean truth was loaded only after all four candidate-value vectors were locked.','No source page, mask, model, proposal, threshold, or correction output was modified.','','Primary outputs:','- tables/arm_summary.csv','- tables/paired_2x2_effects_page_bootstrap.csv','- tables/expanded_groups_2x2_effects.csv','- tables/page_harm_tradeoff_flags.csv','- visual_audit/','- notebook70_verdict.json']
(OUT/'executive_summary.txt').write_text(chr(10).join(map(str,lines)))

required=[OUT/'executive_summary.txt',OUT/'notebook70_verdict.json',OUT/'experiment_identity.json',AUDIT/'read_only_and_truth_firewall_audit.json',AUDIT/'page_source_identity.csv',AUDIT/'clean_truth_identity_after_lock.csv',AUDIT/'truth_firewall.json',LOCK/'candidate_lock.json',LOCK/'candidate_value_inventory.csv',TABLES/'arm_summary.csv',TABLES/'paired_2x2_effects_page_bootstrap.csv',TABLES/'expanded_groups_2x2_effects.csv',TABLES/'metrics_by_page_arm.csv',TABLES/'page_harm_tradeoff_flags.csv',TABLES/'visual_audit_manifest.csv']
for p in required: require(p.exists(),f'Missing final artefact {p}'); shutil.copy2(p,UPLOAD/p.name)
readme = 'Notebook 70 curated 2x2 crossover evidence. Heavy visual PNGs remain in the durable run visual_audit directory and are referenced by visual_audit_manifest.csv. No production authority.' + chr(10)
(UPLOAD/'README.txt').write_text(readme)
rows=[]
for p in sorted(UPLOAD.iterdir()):
    if p.is_file(): rows.append({'file':p.name,'bytes':p.stat().st_size,'sha256':sha256_file(p)})
atomic_csv(pd.DataFrame(rows),UPLOAD/'upload_manifest.csv')
shutil.make_archive(str(OUT/'notebook70_upload_bundle'),'zip',UPLOAD)
print(chr(10).join(map(str,lines))); print('Bundle:',OUT/'notebook70_upload_bundle.zip')

## Outputs to inspect first
1. `executive_summary.txt`
2. `tables/arm_summary.csv`
3. `tables/paired_2x2_effects_page_bootstrap.csv`
4. `tables/expanded_groups_2x2_effects.csv`
5. `tables/page_harm_tradeoff_flags.csv`
6. `visual_audit/`
7. `notebook70_verdict.json`

**Next question:** Do the clean-truth results show that 59R5 grouping and the V8.8 correction engine provide independent/additive benefits, or is one effect negligible once genuine-print loss and residual annotation are measured separately?